# Import libraries

In [44]:
# Libraries
# Data manipulation and storage
import pandas as pd
import numpy as np
import anndata as ad
import re

# Plotting
import seaborn as sns
import matplotlib.pyplot as plt

# Import packages for neuron visualizations
from brainrender import Scene
from brainrender.actors import Points
import vedo
vedo.settings.default_backend= 'vtk'

# Transformations, grid search and cross validation
from sklearn.preprocessing import StandardScaler  
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline 
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_predict

# Classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

# For exporting best classifier to another notebook:
import joblib
from sklearn.base import clone

# ignore warnings:
import warnings
from sklearn.exceptions import UndefinedMetricWarning

warnings.filterwarnings(
    "ignore",
    category=UndefinedMetricWarning
)

# Load and transform data into required format

In [45]:
# load EC data (imputed MERFISH) object
source = "Data/ec_obj_imputed_log2.h5ad"
ec_data = ad.read_h5ad(source)

# Load SNR data for visualisations on Brain Render
ec_snr_data = pd.read_csv('Data/master_detailed_comment.csv')

# Load in the metadata to get access to location, sructure(region) and classes (inc subclasses and supertypes)
cell_metadata = ec_data.obs

# Tables 6 and 7

In [46]:
# Decide which locations and which supertypes to filter for
# Filters for supertypes:
all_supertypes = cell_metadata["supertype"].astype(str).unique()

non_gaba_supertypes = [
    s for s in all_supertypes
    if "gaba" not in s.lower()
]

layer5_supertypes = [
    s for s in non_gaba_supertypes
    if re.search(r"\bL5\b|\bL5/6\b|\bL4/5\b", s)
]

no_layer_supertypes = [
    s for s in non_gaba_supertypes
    if not re.search(r"\bL\d", s)
]

supertype_filter = layer5_supertypes + no_layer_supertypes

# Filter for only supertypes that the client is interested in
cell_metadata = cell_metadata.loc[cell_metadata['supertype'].isin(supertype_filter)]
cell_metadata = cell_metadata.copy()
cell_metadata["supertype"] = cell_metadata["supertype"].cat.remove_unused_categories()

# A list of the 4 supertypes the client has stated definitely appear in layer5a
directed_supertypes = ['0129 NP PPP Glut_1','0007 L5/6 IT TPE-ENT Glut_1','0010 L5/6 IT TPE-ENT Glut_4','0008 L5/6 IT TPE-ENT Glut_2']

print(cell_metadata['supertype'].value_counts().sum())
print(len(cell_metadata['supertype'].value_counts()))

top_9_supertypes = (
    cell_metadata["supertype"]
    .value_counts()
    .head(9)
    .index
    .tolist()
)
print(top_9_supertypes)

4567
47
['0010 L5/6 IT TPE-ENT Glut_4', '0008 L5/6 IT TPE-ENT Glut_2', '0007 L5/6 IT TPE-ENT Glut_1', '0067 ENTmv-PA-COAp Glut_2', '0004 IT EP-CLA Glut_2', '0068 ENTmv-PA-COAp Glut_3', '0135 HPF CR Glut_1', '0066 ENTmv-PA-COAp Glut_1', '0129 NP PPP Glut_1']


In [47]:
cell_metadata.columns


Index(['brain_section_label', 'class', 'subclass', 'supertype', 'structure',
       'substructure', 'x_ccf', 'y_ccf', 'z_ccf'],
      dtype='object')

In [48]:
cell_metadata.to_csv("Data/MERFISH_metadata_filtered.csv")

# Figure 11

In [32]:
# Filter for chosen locations and chosen classes

# cell_metadata = cell_metadata.loc[cell_metadata['substructure'].isin(location_filter)]
cell_metadata = cell_metadata.loc[cell_metadata['supertype'].isin(supertype_filter)]

# Create an (x,y,z) object to classify each Merfish point into one of the clusters found previously
X_merfish = cell_metadata[['x_ccf','y_ccf','z_ccf', 'supertype']]

X_merfish_mirror = X_merfish.copy()

# Separating points on left and right side of brain as classifier only works on 
right_mask = X_merfish["z_ccf"] < 5700
left_mask = X_merfish["z_ccf"] > 5700
X_merfish_mirror.loc[right_mask, "z_ccf"] = (
    2 * 5700 - X_merfish_mirror.loc[right_mask, "z_ccf"]
)




In [42]:

aec_cells = X_merfish_mirror
aec_coords = np.array(aec_cells[["x_ccf", "y_ccf", "z_ccf"]])
aec_points = Points(aec_coords, radius=20, colors="red", alpha=0.5)



xec_cells = ec_snr_data
xec_coords = np.array(xec_cells[["x", "y", "z"]])
xec_points = Points(xec_coords, radius=30, colors="black")


scene = Scene(
    atlas_name="allen_mouse_25um",
    title="SNR and MERFISH offset"
)

scene.add(aec_points)
scene.add(xec_points)

scene.add_brain_region("ENTl", alpha=0.1, color="blue")
scene.add_brain_region("ENTm", alpha=0.1, color="blue")

# cam = scene.plotter.camera

# cam.SetPosition(6610.21, -31614.6, -6569.89)
# cam.SetFocalPoint(7829.74, 4296.03, -5694.5)
# cam.SetViewUp(-0.999424, 0.0339266, 0.000575096)
# cam.SetClippingRange(25503.4, 46935.6)
# cam.SetViewAngle(18.75)

scene.render(camera='top')
scene.plotter.screenshot("Figures/fig_11_top_view.png")

In [43]:

aec_cells = X_merfish_mirror
aec_coords = np.array(aec_cells[["x_ccf", "y_ccf", "z_ccf"]])
aec_points = Points(aec_coords, radius=20, colors="red", alpha=0.5)



xec_cells = ec_snr_data
xec_coords = np.array(xec_cells[["x", "y", "z"]])
xec_points = Points(xec_coords, radius=30, colors="black")


scene = Scene(
    atlas_name="allen_mouse_25um",
    title="SNR and MERFISH offset"
)

scene.add(aec_points)
scene.add(xec_points)

scene.add_brain_region("ENTl", alpha=0.1, color="blue")
scene.add_brain_region("ENTm", alpha=0.1, color="blue")

# cam = scene.plotter.camera

# cam.SetPosition(6610.21, -31614.6, -6569.89)
# cam.SetFocalPoint(7829.74, 4296.03, -5694.5)
# cam.SetViewUp(-0.999424, 0.0339266, 0.000575096)
# cam.SetClippingRange(25503.4, 46935.6)
# cam.SetViewAngle(18.75)

scene.render(camera='sagittal2')
scene.plotter.screenshot("Figures/fig_11_sag_view.png")